In [1]:
import os
import shutil
from collections import defaultdict
import tiktoken  # Helps count tokens accurately

MAX_TOKENS = 8192
tokenizer = tiktoken.get_encoding("cl100k_base")

In [2]:
import json
from openai import OpenAI
import fitz
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import re
from prompts import *
import pandas as pd
import random
from math import ceil

In [7]:
# Define source and destination directories
source_dir = os.path.expanduser("~/Downloads/5 - GSP Scoring Rubrics")
destination_dir = os.path.expanduser("~/Downloads/GSP_Rubric_Drafts")

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Dictionary to track the most recent file for each numbered file
latest_files = defaultdict(lambda: {"path": None, "mtime": 0})

# Traverse all subdirectories
for root, dirs, files in os.walk(source_dir):
    # Check if the folder name contains "Draft"
    if "Draft" in os.path.basename(root):
        for file in files:
            if file.endswith(".xlsx"):
                file_number = "".join(filter(str.isdigit, file))  # Extract the number from the filename
                file_path = os.path.join(root, file)
                file_mtime = os.path.getmtime(file_path)  # Get the last modified time

                # Keep only the most recent version of each numbered file
                if file_number not in latest_files or file_mtime > latest_files[file_number]["mtime"]:
                    latest_files[file_number] = {"path": file_path, "mtime": file_mtime}

# Copy only the latest version of each file to the destination directory
for file_number, file_info in latest_files.items():
    if file_info["path"]:
        destination_path = os.path.join(destination_dir, os.path.basename(file_info["path"]))
        shutil.copy2(file_info["path"], destination_path)
        print(f"Copied: {file_info['path']} -> {destination_path}")

print("File collection and deduplication complete.")

Copied: /Users/ryan/Downloads/5 - GSP Scoring Rubrics/32_PetalumaValley/32_DraftGSP/32_PetalumaValley_DraftGSP_ScoringRubric.xlsx -> /Users/ryan/Downloads/GSP_Rubric_Drafts/32_PetalumaValley_DraftGSP_ScoringRubric.xlsx
Copied: /Users/ryan/Downloads/5 - GSP Scoring Rubrics/12_Cosumnes/12_DraftGSP/12_Cosumnes_DraftGSP_Scoring Rubric.xlsx -> /Users/ryan/Downloads/GSP_Rubric_Drafts/12_Cosumnes_DraftGSP_Scoring Rubric.xlsx
Copied: /Users/ryan/Downloads/5 - GSP Scoring Rubrics/46_Modesto/46_DraftGSP/46_Modesto_DraftGSP_ScoringRubric.xlsx -> /Users/ryan/Downloads/GSP_Rubric_Drafts/46_Modesto_DraftGSP_ScoringRubric.xlsx
Copied: /Users/ryan/Downloads/5 - GSP Scoring Rubrics/4_SouthAmerican/4_DraftGSP/UPDATED_4_SouthAmerican_DraftGSP_ScoringRubric.xlsx -> /Users/ryan/Downloads/GSP_Rubric_Drafts/UPDATED_4_SouthAmerican_DraftGSP_ScoringRubric.xlsx
Copied: /Users/ryan/Downloads/5 - GSP Scoring Rubrics/7_Vina/7_DraftGSP/7_Vina_DraftGSP_ScoringRubric.xlsx -> /Users/ryan/Downloads/GSP_Rubric_Drafts/7_

In [37]:
downloads_path = os.path.expanduser("~/Downloads")
rubrics_folder = os.path.join(downloads_path, "ChatGDE_Draft_Scoring_Rubrics")
csv_output_folder = os.path.join(rubrics_folder, "Converted_CSVs")

# Create folder for converted CSVs if it doesn't exist
os.makedirs(csv_output_folder, exist_ok=True)

# Convert all .xlsx files to .csv
for filename in os.listdir(rubrics_folder):
    if filename.endswith(".xlsx"):
        file_path = os.path.join(rubrics_folder, filename)
        csv_filename = os.path.splitext(filename)[0] + ".csv"
        csv_path = os.path.join(csv_output_folder, csv_filename)

        try:
            df = pd.read_excel(file_path)  # Read Excel file
            df.to_csv(csv_path, index=False, encoding="utf-8")  # Save as CSV
            print(f"Converted: {filename} → {csv_filename}")
        except Exception as e:
            print(f"Error converting {filename}: {e}")

print("✅ All XLSX files converted to CSV.")

Converted: 20_ButteValley_DraftGSPScoringRubric.xlsx → 20_ButteValley_DraftGSPScoringRubric.csv
Converted: 59_SantaYnezRiverValley_Eastern_DraftGSP_ScoringRubric.xlsx → 59_SantaYnezRiverValley_Eastern_DraftGSP_ScoringRubric.csv
Converted: 9_Solano_DraftGSP_ScoringRubric.xlsx → 9_Solano_DraftGSP_ScoringRubric.csv
Converted: 28_Montecito_DraftGSP_ScoringRubric.xlsx → 28_Montecito_DraftGSP_ScoringRubric.csv
Converted: 50_SanLuisObispoValley_DraftGSP_ScoringRubric.xlsx → 50_SanLuisObispoValley_DraftGSP_ScoringRubric.csv
Converted: 31_OjaiValley_DraftGSP_ScoringRubric.xlsx → 31_OjaiValley_DraftGSP_ScoringRubric.csv
Converted: 60_ScottRiverValley_DraftGSP_Scoringrubric.xlsx → 60_ScottRiverValley_DraftGSP_Scoringrubric.csv
Converted: 8_LosMolinos_DraftGSP_ScoringRubric.xlsx → 8_LosMolinos_DraftGSP_ScoringRubric.csv
Converted: 16_Piru_DraftGSP_Scoring Rubric.xlsx → 16_Piru_DraftGSP_Scoring Rubric.csv
Converted: 24_BedfordColdwater_GSP Scoring Rubric.xlsx → 24_BedfordColdwater_GSP Scoring Rubri

In [3]:
no_test = [2, 8, 9, 10, 11, 12, 13, 15, 16, 19, 20, 21, 23, 26, 27, 35, 38, 39, 69]

In [4]:
def HumanRubric(rubric_file):
    """Create data frame of human scoring rubric given csv file of scored rubric. 
       Index represents question number

    Arguments:
        file - filepath name of human scored csv file
    Returns:
        human_rubric_df - df of human scored rubric  
    """
    # Create cleaned DF of questions 
    human_rubric_df = pd.read_csv(rubric_file)
    human_rubric_df = human_rubric_df.iloc[10:, 3:]
    human_rubric_df = human_rubric_df.reset_index()
    human_rubric_df = human_rubric_df.drop('index', axis=1)
    human_rubric_df.columns = human_rubric_df.iloc[0]
    human_rubric_df = human_rubric_df[1:]

    return human_rubric_df

def extract_text_from_pdf(pdf_path):
    """
    Extracts text from a PDF file, preserving page breaks.

    :param pdf_path: The path to the PDF file.
    :return: A string containing the extracted text with page breaks.
    """
    document = fitz.open(pdf_path)
    text = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        text.append(page.get_text("text"))
    
    # Join the text of each page with form feed characters to indicate page breaks
    return '\f'.join(text)

def split_into_pages(text):
    """
    Splits the text into pages.
    
    :param text: The complete text extracted from the PDF.
    :return: A list of pages.
    """
    pages = text.split('\f')
    return pages

def count_tokens(text):
    """Counts tokens in a given text using OpenAI's tokenizer."""
    return len(tokenizer.encode(text))

def split_text_into_chunks(text, max_tokens=MAX_TOKENS):
    """
    Splits text into smaller chunks that fit within the token limit.

    :param text: The text to split.
    :param max_tokens: Maximum number of tokens per chunk.
    :return: A list of text chunks.
    """
    words = text.split()  # Split by spaces (words)
    chunks = []
    current_chunk = []

    for word in words:
        current_chunk.append(word)
        if count_tokens(" ".join(current_chunk)) > max_tokens:
            chunks.append(" ".join(current_chunk[:-1]))  # Save the last chunk before it exceeds limit
            current_chunk = [word]  # Start a new chunk with the last word

    if current_chunk:
        chunks.append(" ".join(current_chunk))  # Add remaining text

    return chunks

def get_embedding(text):
    """
    Fetches a single embedding for a given text chunk using OpenAI's API.
    If text is too long, it will be split and embeddings averaged.
    """
    text = text.strip().replace("\n", " ")

    if not text:
        return np.zeros(3072).tolist()  # embedding dimension for text-embedding-3-large

    chunks = split_text_into_chunks(text, max_tokens=MAX_TOKENS)
    embeddings = []

    for chunk in chunks:
        try:
            response = client.embeddings.create(
                input=[chunk], model="text-embedding-3-large"
            )
            embeddings.append(response.data[0].embedding)
        except Exception as e:
            print(f"Error fetching embedding: {e}")
            embeddings.append(np.zeros(3072).tolist())

    # Average the embeddings if multiple chunks
    averaged_embedding = np.mean(embeddings, axis=0).tolist()

    return averaged_embedding


def get_embeddings(full_text):
    """
    Splits full text into pages and computes embeddings for each page.
    """
    pages = split_into_pages(full_text)
    embeddings = [get_embedding(page) for page in pages]  # Single embedding per page
    
    return pages, embeddings

def find_most_relevant_pages(pages, embeddings, question, top_n=10):
    """
    Finds the most relevant text pages for a given question using embeddings.
    """
    question_embedding = get_embedding(question)
    scores = []

    for page, page_embedding in zip(pages, embeddings):
        score = cosine_similarity([question_embedding], [page_embedding])[0][0]
        scores.append((score, page))

    scores.sort(reverse=True, key=lambda x: x[0])

    most_relevant_pages = [page for _, page in scores[:top_n]]

    return '\n'.join(most_relevant_pages)

def compare_lists(list1, list2):
    differences = []
    if len(list1) != len(list2):
        return None, "Lists are of different lengths and cannot be compared index by index."
    for i in range(len(list1)):
        if list1[i] != list2[i]:
            differences.append(i)
    return differences

def ChatGDE(pages, embeddings, prompts, rubric_file, no_test, gde_answer_function):
    answers = []
    
    # Generate answers based on pages and embeddings
    for i in range(1, 71):
        if i not in no_test:
            section = find_most_relevant_pages(pages, embeddings, prompts[i-1])
            section_and_question = section + prompts[i-1]
            answers.append(gde_answer_function(section_and_question))
    
    # Load rubric and process chat answers
    rubric = HumanRubric(rubric_file)
    chat_answers = rubric['Answer']
    chat_answers = chat_answers.drop(no_test, errors='ignore')
    chat_answers = ['No' if item == 'Somewhat' else item for item in chat_answers]
    chat_answers = ['NotApplicable' if item == 'Not Applicable' else item for item in chat_answers]
    
    # Compare lists and print differences
    differences = compare_lists(answers, chat_answers)
    if differences:
        print("Differences at indices:", differences)
    else:
        print("No differences. Lists are identical.")

    # Print count of differences
    print(f"Number of differences: {len(differences)}")

def extract_yes_probabilities(responses: list) -> list:
    # Define a mapping of confidence descriptions to their corresponding probabilities
    confidence_mapping = {
        "100%": 1.0,
        "85%": 0.85,
        "75%": 0.75,
        "60%": 0.60,
        "50%": 0.50
    }
    
    yes_probabilities = []
    
    for response in responses:
        parts = response.split(', ')
        answer = parts[0]
        confidence_percentage = parts[-1]  # Extract the last part, which should be the percentage
        probability = confidence_mapping.get(confidence_percentage, 0)  # Default to 0 if not found
        
        if answer == "Yes":
            yes_probabilities.append(probability)
        else:
            yes_probabilities.append(1 - probability)
    
    return yes_probabilities

In [4]:
# Set paths
downloads_path = os.path.expanduser("~/Downloads")
input_folder = os.path.join(downloads_path, "ChatGDE_Draft_GSPs")
output_pages_folder = os.path.join(downloads_path, "GSP_Pages")
output_embeddings_folder = os.path.join(downloads_path, "GSP_Embeddings")

# Create output directories if they don’t exist
os.makedirs(output_pages_folder, exist_ok=True)
os.makedirs(output_embeddings_folder, exist_ok=True)

In [5]:
# OpenAI client setup
api_key = os.environ.get('OPENAI_API_KEY') # Replace with your actual key
client = OpenAI(api_key=api_key)

In [22]:
# Process all PDFs in the input folder
for filename in os.listdir(input_folder):
    if filename.endswith(".pdf"):
        pdf_path = os.path.join(input_folder, filename)
        base_name = os.path.splitext(filename)[0]

        print(f"Processing {filename}...")

        # Extract text and compute embeddings
        text = extract_text_from_pdf(pdf_path)
        pages, embeddings = get_embeddings(text)

        # Save pages as a text file
        pages_path = os.path.join(output_pages_folder, f"{base_name}_pages.txt")
        with open(pages_path, "w", encoding="utf-8") as f:
            f.write("\n\n".join(pages))

        # Save embeddings as a JSON file
        embeddings_path = os.path.join(output_embeddings_folder, f"{base_name}_embeddings.json")
        with open(embeddings_path, "w", encoding="utf-8") as f:
            json.dump(embeddings, f)

        print(f"Saved pages to {pages_path}")
        print(f"Saved embeddings to {embeddings_path}")

print("Processing complete!")

Processing 10_Sutter_DraftGSP.pdf...
Saved pages to /Users/ryan/Downloads/GSP_Pages/10_Sutter_DraftGSP_pages.txt
Saved embeddings to /Users/ryan/Downloads/GSP_Embeddings/10_Sutter_DraftGSP_embeddings.json
Processing 25_ElsinoreValley_DraftGSP.pdf...
Saved pages to /Users/ryan/Downloads/GSP_Pages/25_ElsinoreValley_DraftGSP_pages.txt
Saved embeddings to /Users/ryan/Downloads/GSP_Embeddings/25_ElsinoreValley_DraftGSP_embeddings.json
Processing 14_EastContraCoasta_DraftGSP.pdf...
Saved pages to /Users/ryan/Downloads/GSP_Pages/14_EastContraCoasta_DraftGSP_pages.txt
Saved embeddings to /Users/ryan/Downloads/GSP_Embeddings/14_EastContraCoasta_DraftGSP_embeddings.json
Processing 13_Tracy_DraftGSP.pdf...
Saved pages to /Users/ryan/Downloads/GSP_Pages/13_Tracy_DraftGSP_pages.txt
Saved embeddings to /Users/ryan/Downloads/GSP_Embeddings/13_Tracy_DraftGSP_embeddings.json
Processing 18_ShastaValley_DraftGSP.pdf...
Saved pages to /Users/ryan/Downloads/GSP_Pages/18_ShastaValley_DraftGSP_pages.txt
Save

Saved pages to /Users/ryan/Downloads/GSP_Pages/59_SantaYnezRiverValley_Eastern_DraftGSP_pages.txt
Saved embeddings to /Users/ryan/Downloads/GSP_Embeddings/59_SantaYnezRiverValley_Eastern_DraftGSP_embeddings.json
Processing 22_SanGorgonioPass_DraftGSP.pdf...
Saved pages to /Users/ryan/Downloads/GSP_Pages/22_SanGorgonioPass_DraftGSP_pages.txt
Saved embeddings to /Users/ryan/Downloads/GSP_Embeddings/22_SanGorgonioPass_DraftGSP_embeddings.json
Processing 11_Butte_DraftGSP.pdf...
Saved pages to /Users/ryan/Downloads/GSP_Pages/11_Butte_DraftGSP_pages.txt
Saved embeddings to /Users/ryan/Downloads/GSP_Embeddings/11_Butte_DraftGSP_embeddings.json
Processing 62_Yucaipa_DraftGSP.pdf...
Saved pages to /Users/ryan/Downloads/GSP_Pages/62_Yucaipa_DraftGSP_pages.txt
Saved embeddings to /Users/ryan/Downloads/GSP_Embeddings/62_Yucaipa_DraftGSP_embeddings.json
Processing 47_Turlock_DraftGSP.pdf...
Saved pages to /Users/ryan/Downloads/GSP_Pages/47_Turlock_DraftGSP_pages.txt
Saved embeddings to /Users/ryan

In [18]:
# Set path for the input folder
downloads_path = os.path.expanduser("~/Downloads")
rubrics_folder = os.path.join(downloads_path, "ChatGDE_Draft_Scoring_Rubrics/Converted_CSVs")

# Function to process rubric answers
def process_rubric(file_path):
    """
    Reads a scoring rubric CSV and extracts the cleaned human answers.
    """
    rubric = HumanRubric(file_path)  # Load rubric (assuming HumanRubric is defined elsewhere)
    answers = rubric['Answer']  # Extract 'Answer' column
    
    # Drop 'no_test' if it exists (no error if missing)
    answers = answers.drop(no_test, errors='ignore')
    
    # Apply answer transformations
    answers = ['No' if item == 'Somewhat' else item for item in answers]
    answers = ['NotApplicable' if item == 'Not Applicable' else item for item in answers]
    
    return answers

# Initialize list to store all human answers
human_answers = []

# Loop through all CSV files in the rubrics folder
for filename in os.listdir(rubrics_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        file_path = os.path.join(rubrics_folder, filename)
        print(f"Processing {filename}...")  # Logging
        
        try:
            answers = process_rubric(file_path)
            human_answers.extend(answers)  # Append processed answers
        except Exception as e:
            print(f"Error processing {filename}: {e}")

print("Processing complete! Total human answers collected:", len(human_answers))

Processing UPDATED_4_SouthAmerican_DraftGSP_ScoringRubric.csv...
Processing 42_Monterey_DraftGSP_ScoringRubric.csv...
Processing 27_TuleLake_DraftGSP_ScoringRubric.csv...
Processing 46_Modesto_DraftGSP_ScoringRubric.csv...
Processing 34_Enterprise_DraftGSP_ScoringRubric.csv...
Processing 8_LosMolinos_DraftGSP_ScoringRubric.csv...
Processing 14_EastContraCosta_DraftGSP_ScoringRubric.csv...
Processing 28_Montecito_DraftGSP_ScoringRubric.csv...
Processing 7_Vina_DraftGSP_ScoringRubric.csv...
Processing 9_Solano_DraftGSP_ScoringRubric.csv...
Processing 2_EelRiver_DraftGSP_ScoringRubric.csv...
Processing 48_PleasantValley_DraftGSP_ScoringRubric.csv...
Processing 49_WhiteWolf_DraftGSP_ScoringRubric.csv...
Processing 32_PetalumaValley_DraftGSP_ScoringRubric.csv...
Processing 41_ForebayAquifer_DraftGSP_ScoringRubric.csv...
Processing 24_BedfordColdwater_GSP Scoring Rubric.csv...
Processing 16_Piru_DraftGSP_Scoring Rubric.csv...
Processing 25_ElsinoreVally _DraftGSP_ScoringRubric.csv...
Process

In [6]:
def gde_answer(section_and_question: str) -> tuple:
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model= "ft:gpt-4o-2024-08-06:personal:4o-allgsp:BK8WLmJ2",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a skeptical environmental scientist, tasked with answering questions "
                    "about a section from a Groundwater Sustainability Plan (GSP) document. You are required to provide "
                    "your confidence level along with the response. Format your response as 'X, Z' where 'X' is either "
                    "'Yes' or 'No' and 'Z' is your confidence level, which can be one of the following options: "
                    '["Extremely Confident, 100%", "Very Confident, 85%", "Fairly Confident, 75%", "Modest Confidence, 60%", '
                    '"Random Guess, 50%"]. '
                    "Answer the questions objectively, adhering to the provided spectrums."
                    'You should only state you are "Extremely Confident, 100%" if it is irrefutably true that your answer is correct '
                    "according to the Groundwater Sustainability Plan."
                )
            },
            {
                "role": "user",
                "content": section_and_question
            }
        ],
        temperature=0.7,
    )
    extracted = response.choices[0].message.content
    return extracted

In [7]:
# 🔹 Define paths
downloads_path = os.path.expanduser("~/Downloads")
pages_folder = os.path.join(downloads_path, "GSP_Pages")
embeddings_folder = os.path.join(downloads_path, "GSP_Embeddings")
rubric_folder = os.path.join(downloads_path, "ChatGDE_Draft_Scoring_Rubrics/Converted_CSVs")

In [8]:
# 🔹 Load all human answers from rubric files
human_answers = []
gsp_names = []

In [9]:
# 🔹 Function to process a rubric file and extract human answers
def process_rubric(file_path):
    df = pd.read_csv(file_path)  # Read CSV file
    if "Answer" not in df.columns:
        print(f"Skipping {file_path}: No 'Answer' column found.")
        return []
    answers = df["Answer"].astype(str)  # Ensure all values are strings
    answers = answers.drop(no_test, errors="ignore")  # Drop invalid tests
    answers = ["No" if item == "Somewhat" else item for item in answers]
    answers = ["NotApplicable" if item == "Not Applicable" else item for item in answers]
    return answers

In [12]:
for filename in sorted(os.listdir(rubric_folder)):  
    if filename.endswith(".csv"):
        file_path = os.path.join(rubric_folder, filename)
        answers = process_rubric(file_path)
        human_answers.extend(answers)

        # Extract number, first underscore, and text before the second underscore
        match = re.search(r"(\d+_[^_]+)_", filename)
        if match:
            gsp_name = match.group(1) + "_DraftGSP"
            gsp_names.append(gsp_name)
        else:
            print(f"⚠️ Warning: Could not extract GSP name from {filename}, skipping.")

In [10]:
# 🔹 Load precomputed pages and embeddings
gsp_embeddings = {}
gsp_pages = {}

In [13]:
for gsp_name in gsp_names:
    pages_path = os.path.join(pages_folder, f"{gsp_name}_pages.txt")
    embeddings_path = os.path.join(embeddings_folder, f"{gsp_name}_embeddings.json")

    if not os.path.exists(pages_path) or not os.path.exists(embeddings_path):
        print(f"⚠️ Warning: Missing pages or embeddings for {gsp_name}, skipping.")
        continue

    # Load pages
    with open(pages_path, "r", encoding="utf-8") as f:
        gsp_pages[gsp_name] = f.read().split("\n\n")  # Assuming pages are stored as separate paragraphs

    # Load embeddings
    with open(embeddings_path, "r", encoding="utf-8") as f:
        gsp_embeddings[gsp_name] = json.load(f)

print(f"✅ Loaded {len(gsp_embeddings)} precomputed GSP embeddings.")

⚠️ Warning: Missing pages or embeddings for 21_Carpinteria_DraftGSP, skipping.
⚠️ Warning: Missing pages or embeddings for 28_Montecito_DraftGSP, skipping.
⚠️ Warning: Missing pages or embeddings for 48_PleasantValley_DraftGSP, skipping.
⚠️ Warning: Missing pages or embeddings for 8_LosMolinos_DraftGSP, skipping.
⚠️ Warning: Missing pages or embeddings for 4_SouthAmerican_DraftGSP, skipping.
✅ Loaded 60 precomputed GSP embeddings.


In [14]:
for gsp_name in gsp_names:
    pages_path = os.path.join(pages_folder, f"{gsp_name}_pages.txt")
    embeddings_path = os.path.join(embeddings_folder, f"{gsp_name}_embeddings.json")

    if not os.path.exists(pages_path) or not os.path.exists(embeddings_path):
        print(f"⚠️ Warning: Missing pages or embeddings for {gsp_name}, skipping.")
        continue

    # Load pages
    with open(pages_path, "r", encoding="utf-8") as f:
        gsp_pages[gsp_name] = f.read().split("\n\n")  # Assuming pages are stored as separate paragraphs

    # Load embeddings and ensure they are a flat list of numerical vectors

    with open(embeddings_path, "r", encoding="utf-8") as f:
        try:
            embeddings_data = json.load(f)
        except json.JSONDecodeError:
            print(f"⚠️ Error: Could not decode JSON from {embeddings_path}, skipping.")
            continue # Skip if JSON is invalid

    # Initialize placeholder for this GSP
    gsp_embeddings[gsp_name] = None

    if isinstance(embeddings_data, dict) and "data" in embeddings_data:
        # Handle dictionary format { "data": [ { "embedding": [...] }, ... ] }
        # Ensure 'embedding' key exists and value is a list
        processed_embeddings = []
        valid = True
        for i, entry in enumerate(embeddings_data["data"]):
            if isinstance(entry, dict) and "embedding" in entry and isinstance(entry["embedding"], list):
                 processed_embeddings.append(entry["embedding"])
            else:
                 print(f"⚠️ Warning: Invalid entry format at index {i} in dict data for {gsp_name}. Skipping GSP.")
                 valid = False
                 break
        if valid and processed_embeddings: # Check if list is not empty after processing
             gsp_embeddings[gsp_name] = processed_embeddings
        elif valid: # List was processed but resulted in empty
             print(f"⚠️ Warning: Dictionary data processed but resulted in empty embeddings list for {gsp_name}. Skipping GSP.")

    elif isinstance(embeddings_data, list):
        # Handle list format
        if not embeddings_data: # Use 'not list' which is more Pythonic for empty check
            print(f"⚠️ Warning: Empty embeddings list found in {embeddings_path}, skipping.")
            # No need for continue here, gsp_embeddings[gsp_name] remains None
        else:
            # Now we know the list is not empty, check the first element's type
            first_element = embeddings_data[0]

            if isinstance(first_element, list):
                # It's a list of lists. Check if it's nested one level too deep [[],[],[]] vs [[[...]]]
                # Check the *inner* list before indexing it
                if first_element and isinstance(first_element[0], list):
                     # Potentially list of lists of lists: [[ [...], [...] ] , [ [...], [...] ]]
                     # Or list of lists: [ [...], [...] ] where inner is list of floats
                     # We need to be careful assuming the structure just based on the first element.
                     # A safer assumption for flattening might be if *all* elements are lists.
                     if all(isinstance(sublist, list) for sublist in embeddings_data):
                         # Check if it looks like [[ [...]]] (list containing one list of embeddings)
                         if len(embeddings_data) == 1 and all(isinstance(emb, list) for emb in first_element):
                              print(f"ℹ️ Info: Detected single nested list structure for {gsp_name}. Using inner list.")
                              gsp_embeddings[gsp_name] = first_element # Assume [ [emb1, emb2] ] structure
                         # Check if it looks like [ [[...]], [[...]] ] (list of lists of lists - needs flattening)
                         elif all(isinstance(item, list) for item in first_element): # Check deeper nesting
                             print(f"ℹ️ Info: Flattening potentially nested list structure for {gsp_name}.")
                             # This flattens one level [[a,b], [c,d]] -> [a,b,c,d]
                             # or [[[a]], [[b]]] -> [[a], [b]]
                             gsp_embeddings[gsp_name] = [emb for sublist in embeddings_data for emb in sublist]
                         else:
                             # Looks like the correct format already: [ [emb1], [emb2] ]
                              print(f"ℹ️ Info: Assuming standard list of embeddings structure for {gsp_name}.")
                              gsp_embeddings[gsp_name] = embeddings_data
                     else:
                          print(f"⚠️ Warning: Mixed types found in embeddings list for {gsp_name}. Skipping.")

                elif isinstance(first_element, list): # This elif handles the case where first_element[0] is not a list (e.g., it's a float)
                     # Standard format: [ [emb1_vec], [emb2_vec], ... ]
                     # Check if all elements are lists (vectors)
                     if all(isinstance(emb, list) for emb in embeddings_data):
                         print(f"ℹ️ Info: Assuming standard list of embeddings structure for {gsp_name}.")
                         gsp_embeddings[gsp_name] = embeddings_data
                     else:
                         print(f"⚠️ Warning: Non-list item found in embeddings list for {gsp_name}. Skipping.")
                else:
                     # First element is not a list - unexpected format within the list
                     print(f"⚠️ Warning: First element in embeddings list is not a list for {gsp_name}. Skipping.")
            else:
                 # First element is not a list (e.g., list of numbers? Invalid format)
                 print(f"⚠️ Warning: Unexpected format - list does not contain lists for {gsp_name} in {embeddings_path}. Skipping.")
    else:
        # Embeddings data is not a dict or list
        print(f"⚠️ Warning: Embeddings for {gsp_name} in {embeddings_path} are not in expected dictionary or list format. Skipping.")

    # After processing, check if we actually assigned embeddings successfully
    if gsp_embeddings[gsp_name] is None:
        print(f"❌ Error: Failed to process embeddings for {gsp_name}. Skipping GSP.")
        # Remove the key if processing failed completely to avoid issues later
        if gsp_name in gsp_embeddings: del gsp_embeddings[gsp_name]
        if gsp_name in gsp_pages: del gsp_pages[gsp_name] # Also remove pages if embeddings failed
        continue # Explicitly skip to next GSP name

ℹ️ Info: Flattening potentially nested list structure for 10_Sutter_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 11_Butte_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 12_Cosumnes_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 13_Tracy_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 14_EastContraCosta_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 15_Fillmore_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 16_Piru_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 18_Shasta_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 19_SierraValley_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 1_BigValley_DraftGSP.
ℹ️ Info: Flattening potentially nested list structure for 20_ButteValley_DraftGSP.
⚠️ Warning: Missing pages or embeddings for 21_Carpinteria_DraftGSP, skipping.
ℹ️ Info: Flattening potentially nested li

In [15]:
print(f"✅ Loaded {len(gsp_embeddings)} precomputed GSP embeddings.")

✅ Loaded 60 precomputed GSP embeddings.


In [122]:
# 🔹 Run GPT-based predictions for each GSP
rocs_all = []
rocs_by_gsp = {}

In [42]:
screwed_up = ['57_SantaYnezRiverValley_DraftGSP', '58_SantaYnezRiverValley_DraftGSP']

In [18]:
import time
loop_timings = {}
total_start_time = time.perf_counter()

for gsp_name in ['10_Sutter_DraftGSP',
 '11_Butte_DraftGSP',
 '12_Cosumnes_DraftGSP',
 '13_Tracy_DraftGSP',
 '14_EastContraCosta_DraftGSP']:
    # 2. Record the start time for this specific iteration
    start_time = time.perf_counter()
    
    print(f"Processing {gsp_name}...") # Optional: good for long-running loops
    
    gsp_probs_x = []
    for i in range(1, 71):
        if i not in no_test:
            section = find_most_relevant_pages(gsp_pages[gsp_name], gsp_embeddings[gsp_name], prompts[i-1])
            section_and_question = section + prompts[i-1]
            gsp_probs_x.append(gde_answer(section_and_question))
            
    # 3. Record the end time and calculate the duration
    end_time = time.perf_counter()
    duration = end_time - start_time
    
    # 4. Store the duration with its corresponding gsp_name
    loop_timings[gsp_name] = duration

total_end_time = time.perf_counter()

# 5. At the very end, print a summary of how long each step took
print("\n--- Loop Timing Report ---")
for gsp_name, duration in loop_timings.items():
    print(f"Iteration for '{gsp_name}' took {duration:.2f} seconds.")

print("--------------------------")
total_duration = total_end_time - total_start_time
print(f"Total processing time: {total_duration:.2f} seconds.")

Processing 10_Sutter_DraftGSP...
Processing 11_Butte_DraftGSP...
Processing 12_Cosumnes_DraftGSP...
Processing 13_Tracy_DraftGSP...
Processing 14_EastContraCosta_DraftGSP...

--- Loop Timing Report ---
Iteration for '10_Sutter_DraftGSP' took 125.68 seconds.
Iteration for '11_Butte_DraftGSP' took 141.17 seconds.
Iteration for '12_Cosumnes_DraftGSP' took 90.86 seconds.
Iteration for '13_Tracy_DraftGSP' took 82.73 seconds.
Iteration for '14_EastContraCosta_DraftGSP' took 76.82 seconds.
--------------------------
Total processing time: 517.27 seconds.


In [43]:
for gsp_name in gsp_names:
    
    if gsp_name in rocs_by_gsp:
        print(f"⏭️ Skipping {gsp_name}: already processed and found in rocs_by_gsp.")
        continue
        
    if gsp_name not in gsp_embeddings:
        print(f"⚠️ Warning: No embeddings found for {gsp_name}, skipping.")
        continue
    
    if gsp_name in screwed_up:
        print(f"⚠️ Warning: {gsp_name} is screwed up, skipping.")
        continue
        
    print(f"Running GPT predictions for {gsp_name}...")

    gsp_probs = []
    for i in range(1, 71):
        if i not in no_test:
            section = find_most_relevant_pages(gsp_pages[gsp_name], gsp_embeddings[gsp_name], prompts[i-1])
            section_and_question = section + prompts[i-1]
            gsp_probs.append(gde_answer(section_and_question))

    roc_values = extract_yes_probabilities(gsp_probs)
    rocs_by_gsp[gsp_name] = roc_values
    print(f"  -> Completed {gsp_name}. Stored {len(roc_values)} ROC values.")

print(f"✅ GPT model predictions completed.")

⏭️ Skipping 10_Sutter_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 11_Butte_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 12_Cosumnes_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 13_Tracy_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 14_EastContraCosta_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 15_Fillmore_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 16_Piru_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 18_Shasta_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 19_SierraValley_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 1_BigValley_DraftGSP: already processed and found in rocs_by_gsp.
⏭️ Skipping 20_ButteValley_DraftGSP: already processed and found in rocs_by_gsp.
⚠️ Warning: No embeddings found for 21_Carpinteria_DraftGSP, skipping.
⏭️ Skipping 22_SanGorgonioPass_DraftGSP: already processed and found in

In [115]:
rocs_all

[0.85,
 0.85,
 0.85,
 0.85,
 0.15000000000000002,
 0.85,
 0.0,
 0.4,
 0.85,
 0.15000000000000002,
 0.85,
 0.4,
 0.85,
 1.0,
 0.85,
 0.85,
 1.0,
 1.0,
 0.85,
 0.15000000000000002,
 0.85,
 0.6,
 0.85,
 0.85,
 0.6,
 0.15000000000000002,
 0.0,
 0.15000000000000002,
 0.85,
 0.85,
 0.15000000000000002,
 0.5,
 0.25,
 0.25,
 0.85,
 0.25,
 0.85,
 0.75,
 0.85,
 0.15000000000000002,
 0.15000000000000002,
 0.0,
 0.15000000000000002,
 0.15000000000000002,
 0.15000000000000002,
 0.25,
 0.15000000000000002,
 0.0,
 0.85,
 0.85,
 0.0,
 0.6,
 0.85,
 0.85,
 0.15000000000000002,
 0.6,
 0.85,
 0.15000000000000002,
 0.15000000000000002,
 0.85,
 0.4,
 0.85,
 0.15000000000000002,
 0.75,
 0.15000000000000002,
 0.85,
 1.0,
 0.85,
 0.85,
 0.85,
 0.85,
 0.85,
 0.25,
 0.15000000000000002,
 0.15000000000000002,
 0.75,
 0.15000000000000002,
 0.15000000000000002,
 0.25,
 0.85,
 0.85,
 0.15000000000000002,
 0.15000000000000002,
 0.25,
 0.25,
 0.85,
 0.4,
 0.85,
 0.0,
 0.85,
 0.15000000000000002,
 0.4,
 0.85,
 0.150000

In [49]:
len(rocs_by_gsp)

58

In [21]:
def correct_score(df: pd.DataFrame, AreEqual: str='are_equal') -> float:
    """
    Calculates and prints the percentage of chat responses that are correct (equal to human responses).

    Arguments: 
    - df (pd.DataFrame): DataFrame containing the 'AreEqual' column, which represents the comparison 
      between human and chat scores.
    - AreEqual (str, optional): Column name representing the comparison of human and chat scores. 

    Returns:
    - float: The percentage of correct responses by the chat, printed to the console.
    """
    score = round(100* df[AreEqual].sum() / len(df), 2)
    #print(f"{score}% Correctly Answered by Chat")
    return score

def correct_score_no_map(df):
    scored_df = df.copy()
    scored_df.index += 1  #Reset index starting at 1 to align with question numbers
    indexes_to_drop = [8, 9, 10, 11, 12, 13, 15, 16, 19, 20, 21, 23, 26, 27, 38, 39, 69]  # quesionts mentioning mapping
    # Drop the rows and create a new DataFrame
    scored_df = scored_df.drop(indexes_to_drop)
    return scored_df

In [22]:
def accuracy_score_batch(directory_path):
    # Read all files from experiment
    files = [f for f in os.listdir(directory_path) 
            if os.path.isfile(os.path.join(directory_path, f)) and f != '.DS_Store']
    # Determine accuracy score of each plan
    scores = []
    for file in sorted(files):
        print(file)
        df = pd.read_csv(directory_path + '/' + file)
        # Score with map
        score = correct_score(df)
        # Score w/o map questions
        scored_df = correct_score_no_map(df)
        score_no_map = correct_score(scored_df)
        scores.append(score_no_map)
    return scores

In [47]:
output_filename = "gsp_roc_results.json"
with open(output_filename, 'w') as json_file:
        json.dump(rocs_by_gsp, json_file, indent=4) # indent=4 makes the JSON file human-readable
print(f"💾 Successfully saved ROC results to {output_filename}")

💾 Successfully saved ROC results to gsp_roc_results.json


In [11]:
json_filename = "gsp_roc_results.json"
rocs_by_gsp = {} # Initialize as an empty dictionary

print(f"Attempting to load ROC results from: {json_filename}")

with open(json_filename, 'r', encoding='utf-8') as f:
    # Load the dictionary from the JSON file
    rocs_by_gsp = json.load(f)

Attempting to load ROC results from: gsp_roc_results.json


In [15]:
import os
import pandas as pd
import re # Import regular expressions for potentially more robust matching
from sklearn.metrics import roc_auc_score
import numpy as np # Import numpy for NaN handling

# --- Assume your functions are defined ---
# def HumanRubric(rubric_file): ... # Assuming this returns a DataFrame
# (And other necessary functions/variables like no_test exist)

# --- Assume rocs_by_gsp is populated from the previous step ---
# Example: rocs_by_gsp = {'1_BigValley_DraftGSP': [0.8, 0.1, 0.9, ...], ...}

# --- Configuration ---
rubric_folder = 'Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV' # Adjust this path

# --- Data Structures for Results ---
gsp_auc_results = {}
all_gsp_targets = []
all_gsp_probs = []

# --- Main Comparison Loop ---

print(f"Starting comparison against rubrics in: {rubric_folder}")

# 1. Prepare a mapping from numeric prefix to full rubric filename
rubric_files_map = {}
try:
    for r_filename in os.listdir(rubric_folder):
        if r_filename.endswith(".csv"):
            # Extract the leading number(s)
            match = re.match(r"(\d+)[_ ]", r_filename) # Match 1 or more digits followed by _ or space
            if match:
                prefix = match.group(1)
                if prefix in rubric_files_map:
                     print(f"⚠️ Warning: Duplicate prefix '{prefix}' found. File '{r_filename}' might overwrite '{os.path.basename(rubric_files_map[prefix])}'. Check filenames.")
                rubric_files_map[prefix] = os.path.join(rubric_folder, r_filename)
            else:
                print(f"⚠️ Warning: Could not extract numeric prefix from rubric file: {r_filename}. Skipping.")
except FileNotFoundError:
    print(f"❌ Error: Rubric folder not found at '{rubric_folder}'. Cannot proceed.")
    # Handle error appropriately, maybe exit or set results to empty
    gsp_auc_results = {}


# 2. Iterate through GSPs with probabilities
for gsp_name, gsp_probabilities in rocs_by_gsp.items():
    print(f"\nProcessing comparison for: {gsp_name}")

    # Extract numeric prefix from gsp_name
    gsp_match = re.match(r"(\d+)[_ ]", gsp_name)
    if not gsp_match:
        print(f"  -> ⚠️ Warning: Could not extract numeric prefix from GSP name '{gsp_name}'. Skipping.")
        gsp_auc_results[gsp_name] = np.nan # Store NaN for this GSP
        continue
    gsp_prefix = gsp_match.group(1)

    # Find the corresponding rubric file path
    if gsp_prefix not in rubric_files_map:
        print(f"  -> ⚠️ Warning: No rubric file found with prefix '{gsp_prefix}' for GSP '{gsp_name}'. Skipping.")
        gsp_auc_results[gsp_name] = np.nan
        continue

    rubric_path = rubric_files_map[gsp_prefix]
    print(f"  -> Found matching rubric: {os.path.basename(rubric_path)}")

    try:
        # Load and process rubric answers
        # Re-using the user's provided logic structure:
        rubric_df = HumanRubric(rubric_path) # Assumes HumanRubric cleans and resets index correctly
        if 'Answer' not in rubric_df.columns:
             print(f"  -> ⚠️ Error: 'Answer' column not found in {os.path.basename(rubric_path)}. Skipping.")
             gsp_auc_results[gsp_name] = np.nan
             continue

        rubric_answers_raw = rubric_df['Answer']
        # IMPORTANT: Apply the same filtering logic as used when generating probabilities
        # Assuming the index of rubric_answers_raw aligns with 1..70 BEFORE dropping no_test
        rubric_answers_filtered_by_no_test = rubric_answers_raw.drop(index=no_test, errors='ignore')

        # Map values ('Somewhat' -> 'No', 'Not Applicable' -> 'NotApplicable')
        rubric_answers_mapped = ['No' if item == 'Somewhat' else item for item in rubric_answers_filtered_by_no_test]
        rubric_answers_final = ['NotApplicable' if item == 'Not Applicable' else item for item in rubric_answers_mapped]

        # Check if lengths match before filtering NotApplicable
        if len(rubric_answers_final) != len(gsp_probabilities):
            print(f"  -> ⚠️ Error: Length mismatch after initial processing!")
            print(f"     Probabilities count: {len(gsp_probabilities)}")
            print(f"     Rubric answers count (after no_test drop): {len(rubric_answers_final)}")
            print(f"     Skipping AUC calculation for {gsp_name}.")
            gsp_auc_results[gsp_name] = np.nan
            continue

        # Prepare lists for AUC calculation, filtering out 'NotApplicable'
        targets = []  # Ground truth (0 or 1)
        probs = []    # Corresponding probabilities

        for i, answer in enumerate(rubric_answers_final):
            if answer == 'Yes':
                targets.append(1)
                probs.append(gsp_probabilities[i])
            elif answer == 'No':
                targets.append(0)
                probs.append(gsp_probabilities[i])
            # Else (if answer is 'NotApplicable'), skip both target and prob

        # Calculate AUC for this GSP
        if len(targets) < 2:
            print("  -> ⚠️ Warning: Fewer than 2 data points after filtering 'NotApplicable'. Cannot calculate AUC.")
            auc_score = np.nan
        elif len(set(targets)) < 2:
            # Check if all targets are the same (e.g., all 0s or all 1s)
            print(f"  -> ⚠️ Warning: Only one class ({set(targets)}) present after filtering 'NotApplicable'. Cannot calculate AUC.")
            auc_score = np.nan # AUC is undefined in this case
        else:
            try:
                auc_score = roc_auc_score(targets, probs)
                print(f"  -> AUC Score for {gsp_name}: {auc_score:.4f}")
                # Add to overall lists for final calculation
                all_gsp_targets.extend(targets)
                all_gsp_probs.extend(probs)
            except ValueError as e:
                 print(f"  -> ❌ Error calculating AUC for {gsp_name}: {e}. Targets: {set(targets)}, Num points: {len(targets)}")
                 auc_score = np.nan

        gsp_auc_results[gsp_name] = auc_score

    except FileNotFoundError:
        print(f"  -> ❌ Error: Rubric file not found at '{rubric_path}' (should not happen if map is correct). Skipping.")
        gsp_auc_results[gsp_name] = np.nan
    except pd.errors.EmptyDataError:
         print(f"  -> ⚠️ Error: Rubric file '{os.path.basename(rubric_path)}' is empty. Skipping.")
         gsp_auc_results[gsp_name] = np.nan
    except Exception as e: # Catch other potential errors during processing
        print(f"  -> ❌ An unexpected error occurred processing {gsp_name} with {os.path.basename(rubric_path)}: {e}")
        gsp_auc_results[gsp_name] = np.nan


# 3. Calculate Overall AUC
print("\n--- Overall Results ---")
print("Individual GSP AUC Scores:")
for name, score in gsp_auc_results.items():
    if pd.isna(score):
        print(f"  {name}: NaN")
    else:
        print(f"  {name}: {score:.4f}")


overall_auc = np.nan
if not all_gsp_targets:
     print("\nNo data points available for overall AUC calculation.")
elif len(set(all_gsp_targets)) < 2:
     print(f"\nOnly one class ({set(all_gsp_targets)}) present overall. Cannot calculate overall AUC.")
elif len(all_gsp_targets) != len(all_gsp_probs):
     print(f"\n❌ Internal Error: Mismatch between overall targets ({len(all_gsp_targets)}) and probs ({len(all_gsp_probs)}). Cannot calculate overall AUC.")
else:
    try:
        overall_auc = roc_auc_score(all_gsp_targets, all_gsp_probs)
        print(f"\nOverall AUC across all valid GSPs/questions: {overall_auc:.4f}")
    except ValueError as e:
        print(f"\n❌ Error calculating Overall AUC: {e}")


print("\nComparison complete.")

Starting comparison against rubrics in: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV
⚠️ Warning: Could not extract numeric prefix from rubric file: UPDATED_4_SouthAmerican_DraftGSP_ScoringRubric.csv. Skipping.
⚠️ Warning: Could not extract numeric prefix from rubric file: UPDATED_45_SanJacinto_DraftGSP_ScoringRubric.csv. Skipping.
⚠️ Warning: Could not extract numeric prefix from rubric file: UPDATED_52_SanPasqual_DraftGSP_ScoringRubric.csv. Skipping.
⚠️ Warning: Could not extract numeric prefix from rubric file: UPDATED_17_Mound_DraftGSP_ScoringRubric.csv. Skipping.
⚠️ Warning: Could not extract numeric prefix from rubric file: UPDATED_23_SantaMonica_Draft GSP Scoring Rubric.csv. Skipping.
⚠️ Warning: Could not extract numeric prefix from rubric file: UPDATED_55_SantaMargarita_DraftGSP_ScoringRubric.csv. Skipping.

Processing comparison for: 10_Sutter_DraftGSP
  -> Found matching rubric: 10_Sutter_DraftGSP_ScoringRubric.csv
  -> AUC Score for 10_Sutter_DraftGSP: 0.6373

Processing comp

In [10]:
import pandas as pd
import os

# --- Configuration ---
input_folder = 'Downloads/ChatGDE_Draft_Scoring_Rubrics' # Folder containing your .xlsx rubric files
output_folder = 'Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV' # Folder where .csv files will be saved

# --- Dependency Check (Informative) ---
try:
    # Try importing the necessary engine to provide a helpful message if missing
    import openpyxl
except ImportError:
    print("---")
    print("Warning: The 'openpyxl' library is needed by pandas to read .xlsx files.")
    print("If the script fails, please install it using: pip install openpyxl")
    print("---")

# --- Ensure Output Directory Exists ---
try:
    # exist_ok=True prevents an error if the directory already exists
    os.makedirs(output_folder, exist_ok=True)
    print(f"Output directory: {output_folder}")
except OSError as e:
    print(f"❌ Error creating output directory {output_folder}: {e}")
    print("Please check permissions or the path. Exiting.")
    import sys; sys.exit(1) # Exit if we can't create the output folder

# --- Process Files ---
print(f"\nStarting conversion from folder: {input_folder}")
files_processed = 0
files_failed = 0

try:
    all_items = os.listdir(input_folder)
except FileNotFoundError:
    print(f"❌ Error: Input folder not found at '{input_folder}'. Please check the path.")
    all_items = [] # Prevent the loop from running if folder doesn't exist

for item_name in all_items:
    # Construct full path for the potential Excel file
    excel_path = os.path.join(input_folder, item_name)

    # Check if it's a file and has an Excel extension (.xlsx or .xls)
    # Also ignore temporary Excel files that often start with '~$'
    if os.path.isfile(excel_path) and item_name.endswith((".xlsx", ".xls")) and not item_name.startswith('~'):

        # Create the corresponding CSV filename
        base_name = os.path.splitext(item_name)[0] # Get filename without extension
        csv_filename = f"{base_name}.csv"
        csv_path = os.path.join(output_folder, csv_filename)

        print(f"Processing: {item_name}...")
        try:
            # Read the Excel file (usually reads the first sheet by default)
            # If your data is on a different sheet, use: pd.read_excel(excel_path, sheet_name='YourSheetName')
            df = pd.read_excel(excel_path)

            # Write the DataFrame to a CSV file
            # index=False prevents pandas from writing the DataFrame index as a column
            df.to_csv(csv_path, index=False, encoding='utf-8')
            print(f"  ✅ Successfully converted to: {csv_path}")
            files_processed += 1

        except Exception as e:
            # Catch potential errors during reading/writing specific files
            print(f"  ❌ Error processing {item_name}: {e}")
            files_failed += 1
    # Optional: Add an else here to print skipped non-Excel files if desired

# --- Summary ---
print("\n--- Conversion Summary ---")
print(f"Files processed successfully: {files_processed}")
print(f"Files failed: {files_failed}")
print(f"CSV files saved in: {output_folder}")
print("--------------------------")

Output directory: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV

Starting conversion from folder: Downloads/ChatGDE_Draft_Scoring_Rubrics
Processing: 20_ButteValley_DraftGSPScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/20_ButteValley_DraftGSPScoringRubric.csv
Processing: 59_SantaYnezRiverValley_Eastern_DraftGSP_ScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/59_SantaYnezRiverValley_Eastern_DraftGSP_ScoringRubric.csv
Processing: 9_Solano_DraftGSP_ScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/9_Solano_DraftGSP_ScoringRubric.csv
Processing: 28_Montecito_DraftGSP_ScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/28_Montecito_DraftGSP_ScoringRubric.csv
Processing: 50_SanLuisObispoValley_DraftGSP_ScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/50_SanL

  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/25_ElsinoreVally _DraftGSP_ScoringRubric.csv
Processing: 40_Langley_DraftGSP_ScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/40_Langley_DraftGSP_ScoringRubric.csv
Processing: 63_Temescal_DraftGSP_ScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/63_Temescal_DraftGSP_ScoringRubric.csv
Processing: 32_PetalumaValley_DraftGSP_ScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/32_PetalumaValley_DraftGSP_ScoringRubric.csv
Processing: 54_EastBayPlain_DraftGSP_ScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/54_EastBayPlain_DraftGSP_ScoringRubric.csv
Processing: 2_EelRiver_DraftGSP_ScoringRubric.xlsx...
  ✅ Successfully converted to: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV/2_EelRiver_DraftGSP_ScoringRubric.csv
Processing: 47_Turl

In [51]:
import os
import pandas as pd
import re
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np
import json

# --- Load Pre-calculated Probabilities ---
json_filename = "gsp_roc_results.json"
rocs_by_gsp = {}
print(f"Attempting to load ROC results from: {json_filename}")
try:
    with open(json_filename, 'r', encoding='utf-8') as f:
        rocs_by_gsp = json.load(f)
    print(f"✅ Successfully loaded ROC results dictionary from {json_filename}")
except FileNotFoundError:
    print(f"❌ Error: File not found at '{json_filename}'. 'rocs_by_gsp' remains empty.")
except json.JSONDecodeError:
    print(f"❌ Error: Could not decode JSON from '{json_filename}'. Check file content. 'rocs_by_gsp' remains empty.")
except Exception as e:
    print(f"❌ An unexpected error occurred while loading {json_filename}: {e}")

# --- Assume no_test list exists ---
# Example: no_test = [10, 25, 68] # Make sure this is the list of 1-based question numbers to skip

# --- Configuration ---
rubric_folder = 'Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV'

# --- Data Structures for Results ---
gsp_auc_results = {}
gsp_accuracy_results = {}
all_gsp_targets = []
all_gsp_probs = []
all_gsp_predictions = []

# --- Main Comparison Loop ---
print(f"\nStarting comparison against rubrics in: {rubric_folder}")

rubric_files_map = {}
try:
    for r_filename in os.listdir(rubric_folder):
        if r_filename.endswith(".csv") and not r_filename.startswith('~'):
            match = re.match(r"(\d+)[_ ]", r_filename)
            if match:
                prefix = match.group(1)
                if prefix in rubric_files_map:
                    print(f"⚠️ Warning: Duplicate prefix '{prefix}' found. File '{r_filename}' might overwrite '{os.path.basename(rubric_files_map[prefix])}'.")
                rubric_files_map[prefix] = os.path.join(rubric_folder, r_filename)
            else:
                print(f"⚠️ Warning: Could not extract numeric prefix from rubric file: {r_filename}. Skipping.")
except FileNotFoundError:
    print(f"❌ Error: Rubric folder not found at '{rubric_folder}'. Cannot proceed.")
    gsp_auc_results = {}
    gsp_accuracy_results = {}

if not rubric_files_map and os.path.exists(rubric_folder):
     print(f"⚠️ No valid rubric files (.csv with numeric prefix) found in {rubric_folder}.")

for gsp_name, gsp_probabilities in rocs_by_gsp.items():
    print(f"\nProcessing comparison for: {gsp_name}")

    gsp_auc_results[gsp_name] = np.nan
    gsp_accuracy_results[gsp_name] = np.nan

    gsp_match = re.match(r"(\d+)[_ ]", gsp_name)
    if not gsp_match:
        print(f"  -> ⚠️ Warning: Could not extract numeric prefix from GSP name '{gsp_name}'. Skipping.")
        continue
    gsp_prefix = gsp_match.group(1)

    if gsp_prefix not in rubric_files_map:
        print(f"  -> ⚠️ Warning: No rubric file found with prefix '{gsp_prefix}' for GSP '{gsp_name}'. Skipping.")
        continue

    rubric_path = rubric_files_map[gsp_prefix]
    print(f"  -> Found matching rubric: {os.path.basename(rubric_path)}")

    try:
        # *** USING THE PREVIOUS LOGIC BLOCK ***
        rubric_df = HumanRubric(rubric_path) # Assumes HumanRubric returns DF with index matching no_test values
        if 'Answer' not in rubric_df.columns:
             print(f"  -> ⚠️ Error: 'Answer' column not found in DataFrame from {os.path.basename(rubric_path)}. Skipping.")
             # gsp_auc_results[gsp_name] = np.nan # Already set to NaN
             continue

        # Ensure Answer column is string type before processing
        rubric_answers_raw = rubric_df['Answer'].astype(str)

        # IMPORTANT: This assumes the index labels of rubric_answers_raw match the numbers in no_test
        rubric_answers_filtered_by_no_test = rubric_answers_raw.drop(index=no_test, errors='ignore')

        # Map values ('Somewhat' -> 'No', 'Not Applicable' -> 'NotApplicable')
        rubric_answers_mapped = ['No' if item.strip() == 'Somewhat' else item.strip() for item in rubric_answers_filtered_by_no_test]
        rubric_answers_final = ['NotApplicable' if item == 'Not Applicable' else item for item in rubric_answers_mapped]
        # *** END OF PREVIOUS LOGIC BLOCK ***


        # Length check remains the same
        if len(rubric_answers_final) != len(gsp_probabilities):
            print(f"  -> ⚠️ Error: Length mismatch after processing!")
            print(f"     Probabilities count: {len(gsp_probabilities)}")
            print(f"     Rubric answers count (final): {len(rubric_answers_final)}")
            print(f"     Skipping score calculations for {gsp_name}.")
            continue

        # Prepare lists for calculations (remains the same)
        targets = []
        probs = []
        predictions = []
        threshold = 0.5

        for i, answer in enumerate(rubric_answers_final):
            if answer == 'Yes':
                if i < len(gsp_probabilities):
                    targets.append(1)
                    prob = gsp_probabilities[i]
                    probs.append(prob)
                    predictions.append(1 if prob >= threshold else 0)
                else:
                     print(f"  -> ⚠️ Index mismatch (Yes) at index {i}. Skipping.")
            elif answer == 'No':
                if i < len(gsp_probabilities):
                    targets.append(0)
                    prob = gsp_probabilities[i]
                    probs.append(prob)
                    predictions.append(1 if prob >= threshold else 0)
                else:
                     print(f"  -> ⚠️ Index mismatch (No) at index {i}. Skipping.")

        # Calculate AUC (remains the same)
        auc_score = np.nan
        if len(targets) < 2:
            print("  -> ⚠️ Warning: Fewer than 2 data points after filtering for AUC.")
        elif len(set(targets)) < 2:
            print(f"  -> ⚠️ Warning: Only one class ({set(targets)}) present after filtering for AUC.")
        else:
            try:
                auc_score = roc_auc_score(targets, probs)
                print(f"  -> AUC Score for {gsp_name}: {auc_score:.4f}")
            except ValueError as e:
                 print(f"  -> ❌ Error calculating AUC for {gsp_name}: {e}.")
        gsp_auc_results[gsp_name] = auc_score

        # Calculate Accuracy (remains the same)
        acc_score = np.nan
        if not targets:
             print("  -> ⚠️ Warning: No valid data points after filtering for Accuracy.")
        else:
             try:
                 acc_score = accuracy_score(targets, predictions)
                 print(f"  -> Accuracy Score for {gsp_name}: {acc_score:.4f}")
             except ValueError as e:
                  print(f"  -> ❌ Error calculating Accuracy for {gsp_name}: {e}.")
        gsp_accuracy_results[gsp_name] = acc_score

        # Extend overall lists (remains the same)
        if targets:
             all_gsp_targets.extend(targets)
             all_gsp_predictions.extend(predictions)
             if not pd.isna(auc_score) and len(set(targets)) > 1 :
                 all_gsp_probs.extend(probs)

    # Error handling for file reading etc (remains the same)
    except FileNotFoundError:
        print(f"  -> ❌ Error: Rubric file not found at '{rubric_path}'. Skipping.")
    except pd.errors.EmptyDataError:
         print(f"  -> ⚠️ Error: Rubric file '{os.path.basename(rubric_path)}' is empty. Skipping.")
    except KeyError as e:
         print(f"  -> ⚠️ Error: Problem accessing data (maybe missing 'Answer' column or index issue with no_test drop?): {e}")
    except Exception as e:
        print(f"  -> ❌ An unexpected error occurred processing {gsp_name} with {os.path.basename(rubric_path)}: {e}")

# --- Calculate and Print Overall Results --- (remains the same)
print("\n--- Overall Results ---")

print("\nIndividual GSP AUC Scores:")
for name, score in gsp_auc_results.items():
    print(f"  {name}: {score if pd.isna(score) else f'{score:.4f}'}")

print("\nIndividual GSP Accuracy Scores:")
for name, score in gsp_accuracy_results.items():
    print(f"  {name}: {score if pd.isna(score) else f'{score:.4f}'}")

overall_auc = np.nan
if not all_gsp_probs:
     print("\nNo valid data points suitable for overall AUC calculation.")
else:
    num_probs = len(all_gsp_probs)
    auc_targets_overall = [t for i, t in enumerate(all_gsp_targets) if i < num_probs] # Simplistic alignment assumption

    if len(auc_targets_overall) != len(all_gsp_probs):
         print(f"\n❌ Internal Error (AUC Alignment): Mismatch between overall aligned targets ({len(auc_targets_overall)}) and probs ({len(all_gsp_probs)}). Cannot calculate overall AUC.")
    elif len(set(auc_targets_overall)) < 2 :
         print(f"\nOnly one class ({set(auc_targets_overall)}) present overall among data suitable for AUC. Cannot calculate overall AUC.")
    else:
        try:
            overall_auc = roc_auc_score(auc_targets_overall, all_gsp_probs)
            print(f"\nOverall AUC across all valid GSPs/questions (where AUC calculable): {overall_auc:.4f}")
        except ValueError as e:
            print(f"\n❌ Error calculating Overall AUC: {e}")


overall_accuracy = np.nan
if not all_gsp_targets:
     print("\nNo data points available for overall Accuracy calculation.")
elif len(all_gsp_targets) != len(all_gsp_predictions):
     print(f"\n❌ Internal Error (Accuracy): Mismatch between overall targets ({len(all_gsp_targets)}) and predictions ({len(all_gsp_predictions)}). Cannot calculate overall Accuracy.")
else:
    try:
        overall_accuracy = accuracy_score(all_gsp_targets, all_gsp_predictions)
        print(f"\nOverall Accuracy across all valid GSPs/questions: {overall_accuracy:.4f}")
    except ValueError as e:
        print(f"\n❌ Error calculating Overall Accuracy: {e}")

print("\nComparison complete.")

Attempting to load ROC results from: gsp_roc_results.json
✅ Successfully loaded ROC results dictionary from gsp_roc_results.json

Starting comparison against rubrics in: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV

Processing comparison for: 10_Sutter_DraftGSP
  -> Found matching rubric: 10_Sutter_DraftGSP_ScoringRubric.csv
  -> AUC Score for 10_Sutter_DraftGSP: 0.6373
  -> Accuracy Score for 10_Sutter_DraftGSP: 0.6667

Processing comparison for: 11_Butte_DraftGSP
  -> Found matching rubric: 11_Butte_DraftGSP_ScoringRubric.csv
  -> AUC Score for 11_Butte_DraftGSP: 0.7891
  -> Accuracy Score for 11_Butte_DraftGSP: 0.6875

Processing comparison for: 12_Cosumnes_DraftGSP
  -> Found matching rubric: 12_Cosumnes_DraftGSP_Scoring Rubric.csv
  -> AUC Score for 12_Cosumnes_DraftGSP: 0.7461
  -> Accuracy Score for 12_Cosumnes_DraftGSP: 0.6863

Processing comparison for: 13_Tracy_DraftGSP
  -> Found matching rubric: 13_Tracy_GSP Scoring Rubric.csv
  -> ⚠️ Error: 'Answer' column not found in Data

  -> AUC Score for 64_UpperVentura_DraftGSP: 0.8005
  -> Accuracy Score for 64_UpperVentura_DraftGSP: 0.7800

Processing comparison for: 65_BigValley(LakeCounty)_DraftGSP
  -> Found matching rubric: 65_BigValley(LakeCounty)_DraftGSP_ScoringRubric.csv
  -> AUC Score for 65_BigValley(LakeCounty)_DraftGSP: 0.6419
  -> Accuracy Score for 65_BigValley(LakeCounty)_DraftGSP: 0.6600

Processing comparison for: 6_NorthAmerica_DraftGSP
  -> Found matching rubric: 6_NorthAmerica_DraftGSP_ScoringRubric.csv
  -> AUC Score for 6_NorthAmerica_DraftGSP: 0.5750
  -> Accuracy Score for 6_NorthAmerica_DraftGSP: 0.5882

Processing comparison for: 7_Vina_DraftGSP
  -> Found matching rubric: 7_Vina_DraftGSP_ScoringRubric.csv
  -> AUC Score for 7_Vina_DraftGSP: 0.6548
  -> Accuracy Score for 7_Vina_DraftGSP: 0.6078

Processing comparison for: 9_Solano_DraftGSP
  -> Found matching rubric: 9_Solano_DraftGSP_ScoringRubric.csv
  -> AUC Score for 9_Solano_DraftGSP: 0.6482
  -> Accuracy Score for 9_Solano_DraftGSP

In [27]:
import os
import pandas as pd
import re
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve, auc
import numpy as np
import json

json_filename = "gsp_roc_results.json"
rocs_by_gsp = {}
print(f"Attempting to load ROC results from: {json_filename}")
try:
    with open(json_filename, 'r', encoding='utf-8') as f:
        rocs_by_gsp = json.load(f)
    print(f"✅ Successfully loaded ROC results dictionary from {json_filename}")
except FileNotFoundError:
    print(f"❌ Error: File not found at '{json_filename}'. 'rocs_by_gsp' remains empty.")
    # Create dummy rocs_by_gsp if file not found, for script to be runnable
    print("⚠️ 'rocs_by_gsp' is empty. Creating dummy data for demonstration.")
    rocs_by_gsp = {
        "01_GSP_Alpha": [0.1, 0.2, 0.8, 0.9, 0.4],
        "02_GSP_Beta": [0.6, 0.7, 0.3, 0.2, 0.5, 0.99],
        "03 GSP Gamma": [0.5, 0.5, 0.5] # Will test single class warning
    }
    # Save dummy json
    try:
        with open(json_filename, 'w', encoding='utf-8') as f_json_dummy:
            json.dump(rocs_by_gsp, f_json_dummy)
        print(f"✅ Created dummy {json_filename}")
    except Exception as e_json_dummy:
        print(f"❌ Error creating dummy {json_filename}: {e_json_dummy}")

except json.JSONDecodeError:
    print(f"❌ Error: Could not decode JSON from '{json_filename}'. Check file content. 'rocs_by_gsp' remains empty.")
except Exception as e:
    print(f"❌ An unexpected error occurred while loading {json_filename}: {e}")

# IMPORTANT: Ensure your CSV files in this folder are structured as expected by your HumanRubric function.
# Specifically, they need to have at least 11 rows and 4 columns for the iloc[10:, 3:] to work,
# and the 11th row (index 10) from the 4th column (index 3) onwards should contain your desired headers,
# including one named "Answer".
rubric_folder = 'Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV'

# For testing, create a dummy rubric folder if it doesn't exist.
# Note: Dummy CSVs are NOT created here as they need to match HumanRubric's specific parsing.
if not os.path.exists(rubric_folder):
    print(f"⚠️ Rubric folder '{rubric_folder}' not found. Creating dummy folder.")
    os.makedirs(rubric_folder, exist_ok=True)
    print(f"ℹ️ Please place your rubric CSV files (matching HumanRubric's expected format) in '{rubric_folder}'.")

gsp_auc_results = {}
gsp_accuracy_results = {}
gsp_precision_results = {}
gsp_recall_results = {}
gsp_f1_results = {}
gsp_aucpr_results = {}

all_gsp_targets = []
all_gsp_predictions = []
all_gsp_targets_for_roc_aucpr = []
all_gsp_probs_for_roc_aucpr = []

print(f"\nStarting comparison against rubrics in: {rubric_folder}")

rubric_files_map = {}
try:
    for r_filename in os.listdir(rubric_folder):
        if r_filename.endswith(".csv") and not r_filename.startswith('~'):
            match = re.match(r"(\d+)[_ ]", r_filename)
            if match:
                prefix = match.group(1)
                if prefix in rubric_files_map:
                    print(f"⚠️ Warning: Duplicate prefix '{prefix}' found. File '{r_filename}' might overwrite '{os.path.basename(rubric_files_map[prefix])}'.")
                rubric_files_map[prefix] = os.path.join(rubric_folder, r_filename)
            else:
                print(f"⚠️ Warning: Could not extract numeric prefix from rubric file: {r_filename}. Skipping.")
except FileNotFoundError:
    print(f"❌ Error: Rubric folder not found at '{rubric_folder}'. Cannot proceed.")
    gsp_auc_results, gsp_accuracy_results, gsp_precision_results, gsp_recall_results, gsp_f1_results, gsp_aucpr_results = {}, {}, {}, {}, {}, {}

if not rubric_files_map and os.path.exists(rubric_folder):
    print(f"⚠️ No valid rubric files (.csv with numeric prefix) found in {rubric_folder}.")

for gsp_name, gsp_probabilities in rocs_by_gsp.items():
    print(f"\nProcessing comparison for: {gsp_name}")

    gsp_auc_results[gsp_name] = np.nan
    gsp_accuracy_results[gsp_name] = np.nan
    gsp_precision_results[gsp_name] = np.nan
    gsp_recall_results[gsp_name] = np.nan
    gsp_f1_results[gsp_name] = np.nan
    gsp_aucpr_results[gsp_name] = np.nan

    gsp_match = re.match(r"(\d+)[_ ]", gsp_name)
    if not gsp_match:
        print(f"  -> ⚠️ Warning: Could not extract numeric prefix from GSP name '{gsp_name}'. Skipping.")
        continue
    gsp_prefix = gsp_match.group(1)

    if gsp_prefix not in rubric_files_map:
        print(f"  -> ⚠️ Warning: No rubric file found with prefix '{gsp_prefix}' for GSP '{gsp_name}'. Skipping.")
        continue

    rubric_path = rubric_files_map[gsp_prefix]
    print(f"  -> Found matching rubric: {os.path.basename(rubric_path)}")

    try:
        rubric_df = HumanRubric(rubric_path) # Call the provided function

        if 'Answer' not in rubric_df.columns:
             print(f"  -> ⚠️ Error: 'Answer' column not found in DataFrame from {os.path.basename(rubric_path)} (loaded by HumanRubric). Skipping.")
             continue

        # The DataFrame 'rubric_df' returned by HumanRubric has an index that typically starts from 1,
        # corresponding to the rows of data after its internal processing.
        # 'no_test' is assumed to contain 1-based question numbers that match these index values.

        if not rubric_df.index.is_unique:
             print(f"  -> ⚠️ Warning: Index of rubric DataFrame for {os.path.basename(rubric_path)} (from HumanRubric) is not unique. Filtering by 'no_test' might be unpredictable.")

        # Identify valid indices from no_test that exist in the rubric_df's index
        valid_indices_to_drop = [idx for idx in no_test if idx in rubric_df.index]
        if len(valid_indices_to_drop) < len(no_test):
            actual_rubric_indices = set(rubric_df.index)
            ignored_no_test_items = [item for item in no_test if item not in actual_rubric_indices]
            if ignored_no_test_items:
                 print(f"  -> ℹ️ Info: Some no_test items {ignored_no_test_items} not found in rubric index {sorted(list(actual_rubric_indices))[:10]}... for {os.path.basename(rubric_path)}. These items will be ignored for dropping.")
        
        rubric_df_filtered = rubric_df.drop(index=valid_indices_to_drop, errors='ignore')
        rubric_answers_filtered_by_no_test = rubric_df_filtered['Answer'].astype(str)

        # Map values ('Somewhat' -> 'No', 'Not Applicable' -> 'NotApplicable')
        rubric_answers_mapped = ['No' if item.strip().lower() == 'somewhat' else item.strip() for item in rubric_answers_filtered_by_no_test]
        rubric_answers_final = ['NotApplicable' if item.lower() == 'not applicable' else item for item in rubric_answers_mapped]
        
        if len(rubric_answers_final) != len(gsp_probabilities):
            print(f"  -> ⚠️ Error: Length mismatch after processing!")
            print(f"     Probabilities count: {len(gsp_probabilities)}")
            print(f"     Rubric answers count (final, after no_test filter & mapping): {len(rubric_answers_final)}")
            print(f"     Original rubric rows (pre-no_test drop): {len(rubric_df)}")
            print(f"     No_test items to drop: {no_test} (found: {valid_indices_to_drop})")
            print(f"     Skipping score calculations for {gsp_name}.")
            continue

        targets = []
        probs = []
        predictions = []
        threshold = 0.5

        for i, answer in enumerate(rubric_answers_final):
            if answer.lower() == 'yes':
                if i < len(gsp_probabilities): # Check ensures gsp_probabilities is long enough
                    targets.append(1)
                    prob_val = gsp_probabilities[i]
                    probs.append(prob_val)
                    predictions.append(1 if prob_val >= threshold else 0)
            elif answer.lower() == 'no':
                if i < len(gsp_probabilities):
                    targets.append(0)
                    prob_val = gsp_probabilities[i]
                    probs.append(prob_val)
                    predictions.append(1 if prob_val >= threshold else 0)
            # 'NotApplicable' and other values are skipped

        # --- Calculate Metrics ---
        # AUC
        auc_score = np.nan
        if len(targets) < 2: print("  -> ⚠️ Warning: Fewer than 2 data points for AUC.")
        elif len(set(targets)) < 2: print(f"  -> ⚠️ Warning: Only one class ({set(targets)}) for AUC.")
        else:
            try: auc_score = roc_auc_score(targets, probs); print(f"  -> AUC: {auc_score:.4f}")
            except ValueError as e: print(f"  -> ❌ Error AUC: {e}.")
        gsp_auc_results[gsp_name] = auc_score

        # Accuracy, Precision, Recall, F1
        acc_score, precision_val, recall_val, f1_val = np.nan, np.nan, np.nan, np.nan
        if not targets: print("  -> ⚠️ Warning: No valid 'Yes'/'No' data points for thresholded metrics.")
        else:
            try: acc_score = accuracy_score(targets, predictions); print(f"  -> Accuracy: {acc_score:.4f}")
            except ValueError as e: print(f"  -> ❌ Error Accuracy: {e}.")
            try: precision_val = precision_score(targets, predictions, zero_division=0); print(f"  -> Precision: {precision_val:.4f}")
            except ValueError as e: print(f"  -> ❌ Error Precision: {e}.")
            try: recall_val = recall_score(targets, predictions, zero_division=0); print(f"  -> Recall: {recall_val:.4f}")
            except ValueError as e: print(f"  -> ❌ Error Recall: {e}.")
            try: f1_val = f1_score(targets, predictions, zero_division=0); print(f"  -> F1: {f1_val:.4f}")
            except ValueError as e: print(f"  -> ❌ Error F1: {e}.")
        gsp_accuracy_results[gsp_name] = acc_score
        gsp_precision_results[gsp_name] = precision_val
        gsp_recall_results[gsp_name] = recall_val
        gsp_f1_results[gsp_name] = f1_val

        # AUCPR
        aucpr_score = np.nan
        if len(targets) < 2: print("  -> ⚠️ Warning: Fewer than 2 data points for AUCPR.")
        elif len(set(targets)) < 2: print(f"  -> ⚠️ Warning: Only one class ({set(targets)}) for AUCPR.")
        elif not probs: print(f"  -> ⚠️ Warning: Probs list empty for AUCPR.")
        else:
            try:
                precision_curve, recall_curve, _ = precision_recall_curve(targets, probs)
                aucpr_score = auc(recall_curve, precision_curve)
                print(f"  -> AUCPR: {aucpr_score:.4f}")
            except ValueError as e: print(f"  -> ❌ Error AUCPR: {e}.")
            except Exception as e_aucpr: print(f"  -> ❌ Unexpected error AUCPR: {e_aucpr}.")
        gsp_aucpr_results[gsp_name] = aucpr_score

        # Extend overall lists
        if targets:
            all_gsp_targets.extend(targets)
            all_gsp_predictions.extend(predictions)
            if len(set(targets)) > 1 and probs:
                all_gsp_targets_for_roc_aucpr.extend(targets)
                all_gsp_probs_for_roc_aucpr.extend(probs)

    except FileNotFoundError:
        print(f"  -> ❌ Error: Rubric file not found at '{rubric_path}'. Skipping.")
    except pd.errors.EmptyDataError:
         print(f"  -> ⚠️ Error: Rubric file '{os.path.basename(rubric_path)}' is empty. Skipping.")
    except KeyError as e: # e.g. 'Answer' column missing after HumanRubric processing
         print(f"  -> ⚠️ Error: Problem accessing data in {os.path.basename(rubric_path)} (e.g., missing 'Answer' column, or unexpected structure from HumanRubric): {e}")
    except IndexError as e: # Can occur if iloc in HumanRubric fails due to insufficient rows/cols
         print(f"  -> ⚠️ Error: CSV file {os.path.basename(rubric_path)} does not match expected structure for HumanRubric (e.g. not enough rows/columns for slicing): {e}")
    except Exception as e:
        print(f"  -> ❌ An unexpected error occurred processing {gsp_name} with {os.path.basename(rubric_path)}: {e}")
        import traceback
        traceback.print_exc()

print("\n--- Overall Results ---")

def print_metric_dict(name, data_dict):
    print(f"\nIndividual GSP {name} Scores:")
    for gsp, score in data_dict.items():
        print(f"  {gsp}: {score if pd.isna(score) else f'{score:.4f}'}")

print_metric_dict("AUC", gsp_auc_results)
print_metric_dict("Accuracy", gsp_accuracy_results)
print_metric_dict("Precision", gsp_precision_results)
print_metric_dict("Recall", gsp_recall_results)
print_metric_dict("F1", gsp_f1_results)
print_metric_dict("AUCPR", gsp_aucpr_results)

# Overall ROC AUC
overall_auc = np.nan
if not all_gsp_probs_for_roc_aucpr: print("\nNo valid probability data for overall ROC AUC.")
elif len(all_gsp_targets_for_roc_aucpr) != len(all_gsp_probs_for_roc_aucpr): print(f"\n❌ Internal Error (ROC AUC Alignment).")
elif len(set(all_gsp_targets_for_roc_aucpr)) < 2 : print(f"\nOnly one class overall for ROC AUC.")
else:
    try:
        overall_auc = roc_auc_score(all_gsp_targets_for_roc_aucpr, all_gsp_probs_for_roc_aucpr)
        print(f"\nOverall ROC AUC: {overall_auc:.4f}")
    except ValueError as e: print(f"\n❌ Error Overall ROC AUC: {e}")

# Overall Accuracy, Precision, Recall, F1
if not all_gsp_targets: print("\nNo data for overall thresholded metrics.")
elif len(all_gsp_targets) != len(all_gsp_predictions): print(f"\n❌ Internal Error (Thresholded Metrics Alignment).")
else:
    try: print(f"\nOverall Accuracy: {accuracy_score(all_gsp_targets, all_gsp_predictions):.4f}")
    except ValueError as e: print(f"\n❌ Error Overall Accuracy: {e}")
    try: print(f"Overall Precision: {precision_score(all_gsp_targets, all_gsp_predictions, zero_division=0):.4f}")
    except ValueError as e: print(f"\n❌ Error Overall Precision: {e}")
    try: print(f"Overall Recall: {recall_score(all_gsp_targets, all_gsp_predictions, zero_division=0):.4f}")
    except ValueError as e: print(f"\n❌ Error Overall Recall: {e}")
    try: print(f"Overall F1 Score: {f1_score(all_gsp_targets, all_gsp_predictions, zero_division=0):.4f}")
    except ValueError as e: print(f"\n❌ Error Overall F1 Score: {e}")

# Overall AUCPR
overall_aucpr = np.nan
if not all_gsp_probs_for_roc_aucpr: print("\nNo valid probability data for overall AUCPR.")
elif len(all_gsp_targets_for_roc_aucpr) != len(all_gsp_probs_for_roc_aucpr): print(f"\n❌ Internal Error (AUCPR Alignment).")
elif len(set(all_gsp_targets_for_roc_aucpr)) < 2: print(f"\nOnly one class overall for AUCPR.")
else:
    try:
        precision_curve_overall, recall_curve_overall, _ = precision_recall_curve(all_gsp_targets_for_roc_aucpr, all_gsp_probs_for_roc_aucpr)
        overall_aucpr = auc(recall_curve_overall, precision_curve_overall)
        print(f"\nOverall AUCPR: {overall_aucpr:.4f}")
    except ValueError as e: print(f"\n❌ Error Overall AUCPR: {e}")
    except Exception as e_aucpr_overall: print(f"\n❌ Unexpected error Overall AUCPR: {e_aucpr_overall}")

print("\nComparison complete.")

Attempting to load ROC results from: gsp_roc_results.json
✅ Successfully loaded ROC results dictionary from gsp_roc_results.json

Starting comparison against rubrics in: Downloads/ChatGDE_Draft_Scoring_Rubrics_CSV

Processing comparison for: 10_Sutter_DraftGSP
  -> Found matching rubric: 10_Sutter_DraftGSP_ScoringRubric.csv
  -> AUC: 0.6373
  -> Accuracy: 0.6667
  -> Precision: 0.6000
  -> Recall: 0.5714
  -> F1: 0.5854
  -> AUCPR: 0.4183

Processing comparison for: 11_Butte_DraftGSP
  -> Found matching rubric: 11_Butte_DraftGSP_ScoringRubric.csv
  -> AUC: 0.7891
  -> Accuracy: 0.6875
  -> Precision: 0.5200
  -> Recall: 0.8125
  -> F1: 0.6341
  -> AUCPR: 0.7292

Processing comparison for: 12_Cosumnes_DraftGSP
  -> Found matching rubric: 12_Cosumnes_DraftGSP_Scoring Rubric.csv
  -> AUC: 0.7461
  -> Accuracy: 0.6863
  -> Precision: 0.6071
  -> Recall: 0.7727
  -> F1: 0.6800
  -> AUCPR: 0.7535

Processing comparison for: 13_Tracy_DraftGSP
  -> Found matching rubric: 13_Tracy_GSP Scoring R

  -> AUC: 0.6716
  -> Accuracy: 0.6800
  -> Precision: 0.6000
  -> Recall: 0.7143
  -> F1: 0.6522
  -> AUCPR: 0.4492

Processing comparison for: 59_SantaYnezRiverValley_DraftGSP
  -> Found matching rubric: 59_SantaYnezRiverValley_Eastern_DraftGSP_ScoringRubric.csv
  -> AUC: 0.6875
  -> Accuracy: 0.7000
  -> Precision: 0.4737
  -> Recall: 0.6429
  -> F1: 0.5455
  -> AUCPR: 0.3212

Processing comparison for: 5_Colusa_DraftGSP
  -> Found matching rubric: 5_Colusa_DraftGSP_ScoringRubric.csv
  -> AUC: 0.6611
  -> Accuracy: 0.6471
  -> Precision: 0.5600
  -> Recall: 0.6667
  -> F1: 0.6087
  -> AUCPR: 0.3848

Processing comparison for: 60_ScottRiverValley_DraftGSP
  -> Found matching rubric: 60_ScottRiverValley_DraftGSP_Scoringrubric.csv
  -> AUC: 0.7375
  -> Accuracy: 0.6800
  -> Precision: 0.3750
  -> Recall: 0.9000
  -> F1: 0.5294
  -> AUCPR: 0.2333

Processing comparison for: 61_Ukiah_DraftGSP
  -> Found matching rubric: 61_Ukiah_GSP Scoring Rubric.csv
  -> AUC: 0.8222
  -> Accuracy: 0.68